In [ ]:
# # Transformers installation
# ! pip install transformers datasets evaluate accelerate
# # To install from source instead of the last release, comment the command above and uncomment the following one.
# # ! pip install git+https://github.com/huggingface/transformers.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


# Prompt engineering

Prompt engineering or prompting, uses natural language to improve large language model (LLM) performance on a variety of tasks. A prompt can steer the model towards generating a desired output. In many cases, you don't even need a [fine-tuned](#finetuning) model for a task. You just need a good prompt.



## Basics of Prompting in Different Tasks

### Text Generation

Try basic text generation using decoder-only model (GPT2)

In [ ]:
from transformers import pipeline
import torch

torch.manual_seed(0)
generator = pipeline('text-generation', model = 'openai-community/gpt2')
prompt = "Hello, I'm a language model"
generator(prompt, max_length = 30)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Hello, I'm a language model. I'm a kind of a math model. I'm a sort of a cognitive model. I'm a sort of a neuromodel. I'm a sort of a human.\n\nWhat's your favorite part of writing a book?\n\nI love the part where you're writing about one of those experiences where you're trying to sort out what it is that you're trying to do. Or you're trying to sort out how to talk about what you're trying to do. That's what I love about writing books. I love the part where you're trying to sort out what it is that you're trying to do. Or you're trying to sort out how to talk about what you're trying to do. That's what I love about writing books. I love the part where you're trying to sort out what it is that you're trying to do. Or you're trying to sort out how to talk about what you're trying to do. That's what I love about writing books.\n\nDo you have a favorite book you're doing?\n\nI love the part where you're trying to figure out what it is that you're trying to do. Or you

### Text Classification

Try prompting a LLM to classify some text. When you create a prompt, it's important to provide very specific instructions about the task and what the result should look like.


In [ ]:
from transformers import pipeline, AutoTokenizer
import torch


# Load a model for text generation
torch.manual_seed(0)
model = "tiiuae/falcon-7b-instruct"


tokenizer = AutoTokenizer.from_pretrained(model)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

prompt = """Classify the text into neutral, negative or positive.
Text: This movie is definitely one of my favorite movies of its kind. The interaction between respectable and morally strong characters is an ode to chivalry and the honor code amongst thieves and policemen.
Sentiment:
"""

outputs = pipe(prompt, max_new_tokens=10)
for output in outputs:
    print(f"Result: {output['generated_text']}")


# Result: Classify the text into neutral, negative or positive.
# Text: This movie is definitely one of my favorite movies of its kind. The interaction between respectable and morally strong characters is an ode to chivalry and the honor code amongst thieves and policemen.
# Sentiment:
# Positive

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Result: Classify the text into neutral, negative or positive.
Text: This movie is definitely one of my favorite movies of its kind. The interaction between respectable and morally strong characters is an ode to chivalry and the honor code amongst thieves and policemen.
Sentiment:
Positive


Using Flan-T5 model for Text Classification

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Define the input text and the classification task using a prompt
#input_text = "Classify the sentiment of the following review as positive, negative, or neutral: 'I love this product, it works perfectly!'"
input_text = "Classify the sentiment of the following review as positive, negative, or neutral: 'This movie is definitely one of my favorite movies of its kind. The interaction between respectable and morally strong characters is an ode to chivalry and the honor code amongst thieves and policemen.'"

# Tokenize the input text
# return_tensors="pt" specifies PyTorch tensors
inputs = tokenizer(input_text, return_tensors="pt")

# Generate the output (the predicted class label)
outputs = model.generate(**inputs, max_new_tokens=50) # Set max_new_tokens for generation length

# Decode the output tokens into a human-readable string
output_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)

print(f"Input: {input_text}")
print(f"Output: {output_text[0]}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Input: Classify the sentiment of the following review as positive, negative, or neutral: 'This movie is definitely one of my favorite movies of its kind. The interaction between respectable and morally strong characters is an ode to chivalry and the honor code amongst thieves and policemen.'
Output: positive


### Translation

Another task LLMs can perform is translation. We’ll keep using Falcon-7b-instruct, which does a decent job. Once again, here’s how you can write a basic prompt to instruct a model to translate a piece of text from English to Italian:


In [ ]:
torch.manual_seed(2)

prompt = """Translate the English text to Italian.
Text: Sometimes, I've believed as many as six impossible things before breakfast.
Translation:
"""

sequences = pipe(
    prompt,
    max_new_tokens=20,
    do_sample=True,
    top_k=10,
    return_full_text = False,
)

for seq in sequences:
    print(f"{seq['generated_text']}")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_k', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A volte, ho creduto a sei impossibili cose prima di colazione.


Translate back to English (Google Translate): Sometimes, I've believed six impossible things before breakfast.

Using Flan-T5 model for Translation

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# 1. Load model and tokenizer
model_name = "google/flan-t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# 2. Prepare prompt
prompt = "translate English to Italian: Sometimes, I've believed as many as six impossible things before breakfast."
inputs = tokenizer(prompt, return_tensors="pt")

# 3. Generate output
outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Ma avevo pensato il numero di sei esempi


Translate back to English (Google Translate): But I had thought of the number of six examples

**Question:** Given the output from "Falcon-7b-instruct" and "Flan-T5", are these translations both correct?

**answer:**

### In practice 1:

Use our previous flight sentiment data for prompting-based sentiment analysis.

1. Sample top 200 negative and positive tweets as a test set
2. Shuffle these tweets
3. Use Flan-T5-small to make predictions
4. Calculate the precision, recall, and F1 score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.metrics import f1_score

In [ ]:
# Change your own file path
tweetpath = '/content/drive/MyDrive/Course/LIS501/Lab_wk9/flight_sentiment.tsv'
tweets = pd.read_csv(tweetpath, sep ='\t')
print(tweets.shape)
tweets.head()

(14640, 5)


,airline_sentiment,negativereason,retweet_count,airline,text
0,neutral,NaN,0,Virgin America,@VirginAmerica What @dhepburn said.
1,positive,NaN,0,Virgin America,@VirginAmerica plus you've added commercials t...
2,neutral,NaN,0,Virgin America,@VirginAmerica I didn't today... Must mean I n...
3,negative,Bad Flight,0,Virgin America,@VirginAmerica it's really aggressive to blast...
4,negative,Can't Tell,0,Virgin America,@VirginAmerica and it's a really big bad thing...


In [ ]:
positive_tweets = tweets.loc[tweets.airline_sentiment == 'positive', :]
negative_tweets = tweets.loc[tweets.airline_sentiment == 'negative', :]

sample400_tweets = pd.concat([positive_tweets.iloc[0:200, : ], negative_tweets.iloc[0: 200, : ]], axis = 'rows')

sample400_tweets_shuffle = sample400_tweets.sample(frac = 1)

print(sample400_tweets_shuffle.shape)
sample400_tweets_shuffle.head()

(400, 5)


,airline_sentiment,negativereason,retweet_count,airline,text
47,positive,NaN,0,Virgin America,@VirginAmerica wow this just blew my mind
99,negative,Customer Service Issue,0,Virgin America,@VirginAmerica is anyone doing anything there ...
184,positive,NaN,0,Virgin America,@VirginAmerica Only way to fly! #Elevate #Gold
40,positive,NaN,0,Virgin America,"@VirginAmerica View of downtown Los Angeles, t..."
496,negative,Customer Service Issue,0,Virgin America,@VirginAmerica my group got their Cancelled Fl...


In [ ]:
from transformers import pipeline, AutoTokenizer
import torch

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

pred_labels = []
for txt in sample400_tweets_shuffle['text'].tolist():
  prompt = "Classify the sentiment of the following review as positive, negative, or neutral: \'" + txt + "\'"
  inputs = tokenizer(prompt, return_tensors="pt")
  outputs = model.generate(**inputs, max_new_tokens=50)
  output_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)

  pred_labels.append(output_text[0])

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
actual_labels = sample400_tweets_shuffle['airline_sentiment'].tolist()

print(len(pred_labels), '\t', len(actual_labels))

400 	 400


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

pred_labels_number = [1 if i == 'positive' else 0 for i in pred_labels]
actual_labels_number = [1 if i == 'positive' else 0 for i in actual_labels]


# Calculate individual metrics (for binary classification, default pos_label=1)
precision = precision_score(actual_labels_number, pred_labels_number)
recall = recall_score(actual_labels_number, pred_labels_number)
f1 = f1_score(actual_labels_number, pred_labels_number)

print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

# Alternatively, get all metrics and more in a formatted report
print("\nClassification Report:")
print(classification_report(actual_labels_number, pred_labels_number))

Precision: 0.8873239436619719
Recall: 0.945
F1 Score: 0.9152542372881356

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.88      0.91       200
           1       0.89      0.94      0.92       200

    accuracy                           0.91       400
   macro avg       0.91      0.91      0.91       400
weighted avg       0.91      0.91      0.91       400



### In practice 2:

Use "Falcon-7b-instruct" and "Flan-T5" for other prompting-based translation cases (English to other langauges), and compare your results.

Try 3 different cases, and share your results.

The challenge lies in designing prompts that produces the results you're expecting because language is so incredibly nuanced and expressive.

This guide covers prompt engineering best practices, techniques, and examples for how to solve language and reasoning tasks.

## Best practices

1. Try to pick the latest models for the best performance. Keep in mind that LLMs can come in two variants, [base](https://hf.co/mistralai/Mistral-7B-v0.1) and [instruction-tuned](https://hf.co/mistralai/Mistral-7B-Instruct-v0.1) (or chat).

    Base models are excellent at completing text given an initial prompt, but they're not as good at following instructions. Instruction-tuned models are specifically trained versions of the base models on instructional or conversational data. This makes instruction-tuned models a better fit for prompting.

    > [!WARNING]
    > Modern LLMs are typically decoder-only models, but there are some encoder-decoder LLMs like [Flan-T5](https://huggingface.co/docs/transformers/main/en/tasks/../model_doc/flan-t5) or [BART](https://huggingface.co/docs/transformers/main/en/tasks/../model_doc/bart) that may be used for prompting. Load these models directly with the [AutoModelForSeq2SeqLM](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoModelForSeq2SeqLM) class (instead of using [Pipeline](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.Pipeline)) and generate outputs from the model itself.

2. Start with a short and simple prompt, and iterate on it to get better results.

3. Put instructions at the beginning or end of a prompt. For longer prompts, models may apply optimizations to prevent attention from scaling quadratically, which places more emphasis at the beginning and end of a prompt.

4. Clearly separate instructions from the text of interest.

5. Be specific and descriptive about the task and the desired output, including for example, its format, length, style, and language. Avoid ambiguous descriptions and instructions.

6. Instructions should focus on "what to do" rather than "what not to do".

7. Lead the model to generate the correct output by writing the first word or even the first sentence.

8. Try other techniques like [few-shot](#few-shot) and [chain-of-thought](#chain-of-thought) to improve results.

9. Test your prompts with different models to assess their robustness.

10. Version and track your prompt performance.

## Techniques

Crafting a good prompt alone, also known as zero-shot prompting, may not be enough to get the results you want. You may need to try a few prompting techniques to get the best performance.

This section covers a few prompting techniques.

### Few-shot prompting

Few-shot prompting improves accuracy and performance by including specific examples of what a model should generate given an input. The explicit examples give the model a better understanding of the task and the output format you're looking for. Try experimenting with different numbers of examples (2, 4, 8, etc.) to see how it affects performance. The example below provides the model with 1 example (1-shot) of the output format (a date in MM/DD/YYYY format) it should return.

In [ ]:
from transformers import pipeline, AutoTokenizer
import torch

torch.manual_seed(5)
prompt = """Text: The first human went into space and orbited the Earth on April 12, 1961.
Date: 04/12/1961
Text: The first-ever televised presidential debate in the United States took place on September 28, 1960, between presidential candidates John F. Kennedy and Richard Nixon.
Date:"""


model = "tiiuae/falcon-7b-instruct"


tokenizer = AutoTokenizer.from_pretrained(model)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

sequences = pipe(
    prompt,
    max_new_tokens=8,
    do_sample=True,
    top_k=10,
)

for seq in sequences:
    print(f"Result: {seq['generated_text']}")

# Result: Text: The first human went into space and orbited the Earth on April 12, 1961.
# Date: 04/12/1961
# Text: The first-ever televised presidential debate in the United States took place on September 28, 1960, between presidential candidates John F. Kennedy and Richard Nixon.
# Date: 09/28/1960

The downside of few-shot prompting is that you need to create lengthier prompts which increases computation and latency. There is also a limit to prompt lengths. Finally, a model can learn unintended patterns from your examples, and it may not work well on complex reasoning tasks.

To improve few-shot prompting for modern instruction-tuned LLMs, use a model's specific [chat template](https://huggingface.co/docs/transformers/main/en/tasks/../conversations). These models are trained on datasets with turn-based conversations between a "user" and "assistant". Structuring your prompt to align with this can improve performance.

Structure your prompt as a turn-based conversation and use the `apply_chat_template` method to tokenize and format it.

In [ ]:
from transformers import pipeline
import torch

model = "tiiuae/falcon-7b-instruct"
pipe = pipeline(model=model, dtype=torch.bfloat16, device_map="auto")

messages = [
    {"role": "user", "content": "Text: The first human went into space and orbited the Earth on April 12, 1961."},
    {"role": "assistant", "content": "Date: 04/12/1961"},
    {"role": "user", "content": "Text: The first-ever televised presidential debate in the United States took place on September 28, 1960, between presidential candidates John F. Kennedy and Richard Nixon."}
]

prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

outputs = pipe(prompt, max_new_tokens=12, do_sample=True, top_k=10)

for output in outputs:
    print(f"Result: {output['generated_text']}")

While the basic few-shot prompting approach embedded examples within a single text string, the chat template format offers the following benefits.

- The model may have a potentially improved understanding because it can better recognize the pattern and the expected roles of user input and assistant output.
- The model may more consistently output the desired output format because it is structured like its input during training.

Always consult a specific instruction-tuned model's documentation to learn more about the format of their chat template so that you can structure your few-shot prompts accordingly.

### Chain-of-thought

Chain-of-thought (CoT) is effective at generating more coherent and well-reasoned outputs by providing a series of prompts that help a model "think" more thoroughly about a topic.

The example below provides the model with several prompts to work through intermediate reasoning steps.

In [ ]:
from transformers import pipeline
import torch


model = "tiiuae/falcon-7b-instruct"
pipe = pipeline(model=model, dtype=torch.bfloat16, device_map="auto")

prompt = """Let's go through this step-by-step:
1. You start with 15 muffins.
2. You eat 2 muffins, leaving you with 13 muffins.
3. You give 5 muffins to your neighbor, leaving you with 8 muffins.
4. Your partner buys 6 more muffins, bringing the total number of muffins to 14.
5. Your partner eats 2 muffins, leaving you with 12 muffins.
If you eat 6 muffins, how many are left?"""

outputs = pipe(prompt, max_new_tokens=20, do_sample=True, top_k=10)
for output in outputs:
    print(f"Result: {output['generated_text']}")

# Result: Let's go through this step-by-step:
# 1. You start with 15 muffins.
# 2. You eat 2 muffins, leaving you with 13 muffins.
# 3. You give 5 muffins to your neighbor, leaving you with 8 muffins.
# 4. Your partner buys 6 more muffins, bringing the total number of muffins to 14.
# 5. Your partner eats 2 muffins, leaving you with 12 muffins.
# If you eat 6 muffins, how many are left?
# Answer: 6

Like [few-shot](#few-shot) prompting, the downside of CoT is that it requires more effort to design a series of prompts that help the model reason through a complex task and prompt length increases latency.

</hfoption>
</hfoptions>